In [0]:
import json
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.utils import AnalysisException

class SAPSmokeTester:
    """Core logic for running smoke tests."""
    def __init__(self, spark, config):
        self.spark = spark
        self.config = config
        self.catalog = config['settings']['catalog']
        self.schema = config['settings']['schema']
        self.results = []
        
    def _get_df(self, table_name):
        return self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")

    def log_result(self, table, check_type, status, message, metrics=None):
        self.results.append({
            "Table": table,
            "Check": check_type,
            "Status": status,
            "Message": message,
            "Metrics": metrics
        })
        # Optional: Print simple status to stdout as we go
        # print(f"[{status}] {table} - {check_type}: {message}")

    def run(self):
        print(f"Starting Smoke Test on {self.catalog}.{self.schema}...")
        
        for t_conf in self.config['tables']:
            table_name = t_conf['table']
            try:
                df = self._get_df(table_name)
            except AnalysisException:
                self.log_result(table_name, "Existence", "FAIL", "Table not found")
                continue 

            df.cache() # Optimization
            row_count = df.count()
            
            for check in t_conf['checks']:
                ctype = check['type']
                
                if ctype == 'count':
                    min_r = check.get('min', 1)
                    res = "PASS" if row_count >= min_r else "FAIL"
                    msg = f"Rows: {row_count} (Min: {min_r})"
                    self.log_result(table_name, "Volume", res, msg, row_count)

                elif ctype == 'no_nulls':
                    cols = check['columns']
                    missing = [c for c in cols if c not in df.columns]
                    if missing:
                         self.log_result(table_name, "Schema", "FAIL", f"Missing cols: {missing}")
                         continue
                    
                    null_expr = " OR ".join([f"{c} IS NULL" for c in cols])
                    null_cnt = df.filter(null_expr).count()
                    res = "PASS" if null_cnt == 0 else "FAIL"
                    self.log_result(table_name, "Null Check", res, f"Nulls in {cols}: {null_cnt}", null_cnt)

                elif ctype == 'allowed_values':
                    col, allowed = check['column'], check['values']
                    if col not in df.columns:
                        self.log_result(table_name, "Schema", "FAIL", f"Missing col: {col}")
                        continue
                    
                    bad_cnt = df.filter(~F.col(col).isin(allowed)).count()
                    res = "PASS" if bad_cnt == 0 else "FAIL"
                    self.log_result(table_name, "Validity", res, f"Invalid values in {col}: {bad_cnt}", bad_cnt)

                elif ctype == 'foreign_key':
                    ptable, c_col, p_col = check['parent_table'], check['child_col'], check['parent_col']
                    try:
                        pdf = self._get_df(ptable)
                        orphans = df.join(pdf, df[c_col] == pdf[p_col], "left_anti").count()
                        res = "PASS" if orphans == 0 else "FAIL"
                        self.log_result(table_name, "Integrity", res, f"Orphans in {c_col}: {orphans}", orphans)
                    except Exception as e:
                        self.log_result(table_name, "Integrity", "ERROR", f"Join failed: {str(e)}")

            df.unpersist()
        
        return pd.DataFrame(self.results)

# --- NEW HELPER FUNCTION ---
def execute_smoke_tests(config_path):
    """
    One-stop function to load config, run tests, and return results.
    """
    # 1. Load Config
    try:
        with open(config_path, "r") as f:
            config_data = json.load(f)
    except FileNotFoundError:
        print(f"❌ Critical Error: Config file not found at {config_path}")
        return None

    # 2. Run Tests
    tester = SAPSmokeTester(spark, config_data)
    df_results = tester.run()
    
    # 3. Automatic Status Check
    if "FAIL" in df_results['Status'].values:
        print("\n❌ SMOKE TESTS FAILED")
    else:
        print("\n✅ ALL SMOKE TESTS PASSED")
        
    return df_results